# FloodNet 2025 Data Gap Analysis -- Full Sensor Network

`tidal_analysis.ipynb` found that 74 of ~200 tidal/coastal/floodplain candidate sensors recorded zero flood events in `aq7i-eu5q` for all of 2025, despite being deployed and live, and despite the network overall having record activity that year. This notebook repeats that check against the **entire deployed FloodNet network** (415 sensors, not just the tidal/floodplain subset) to size the problem.

Every raw fetch below sets an explicit `$limit` well above the actual row count and is cross-checked against an independent `count(*)` -- NYC Open Data defaults to a 1,000-row limit with no error on truncation, so this proves nothing was silently cut off.

In [1]:
import os
from urllib.parse import urlencode, quote

import numpy as np
import pandas as pd

os.chdir(r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides")

FLOOD_EVENTS_RESOURCE = 'aq7i-eu5q'

# Small helper so every fetch below reads as pd.read_json(soda_url(...)) instead of
# manually hand-encoding query strings like the %20/%60-escaped URLs elsewhere in this
# project. quote_via=quote keeps spaces as %20 (not urlencode's default '+'), matching
# the encoding Socrata's own UI produces, though either is valid for a query string.
def soda_url(resource, **params):
    return f'https://data.cityofnewyork.us/resource/{resource}.json?' + urlencode(params, quote_via=quote)

## Load the full deployed sensor network

In [2]:
# Full cleaned sensor table from FloodNet_Sensor_Locations.ipynb -- NOT filtered to
# floodplain/coastal/tidal like tidal_analysis.ipynb's final_sensors_to_show. This is
# every currently-deployed FloodNet sensor with an active Directus match.
all_sensors = pd.read_csv('final_deployed_sensors.csv')
all_sensors['date_deployed'] = pd.to_datetime(all_sensors['date_deployed'], errors='coerce')
all_sensors['date_removed'] = pd.to_datetime(all_sensors['date_removed'], errors='coerce')
print(f"Full deployed sensor network: {len(all_sensors)} sensors")

Full deployed sensor network: 415 sensors


## Fetch flood events (2025+), verified against the source

In [3]:
# Same query tidal_analysis.ipynb's cell 6 runs, but nothing here is pre-filtered to
# floodplain/tidal sensors -- every sensor's events are included.
flood_data = pd.read_json(soda_url(
    FLOOD_EVENTS_RESOURCE,
    **{
        '$select': 'sensor_name, sensor_id, flood_start_time, flood_end_time, max_depth_inches, duration_mins',
        '$where': "flood_start_time > '2025-01-01T00:00:00'",
        '$limit': 50000,  # actual row count is ~1.3k; large safety margin, not a real cap
    },
))
flood_data['flood_start_time'] = pd.to_datetime(flood_data['flood_start_time'])
print(f"Fetched {len(flood_data)} flood events (2025-01-01 onward)")

Fetched 1303 flood events (2025-01-01 onward)


In [4]:
# Verification: prove the fetch above wasn't truncated by NYC Open Data's default
# 1,000-row API limit (see the export dialog on the dataset page -- it warns about this
# exact default). count(*) returns one summary row regardless of how much data underlies
# it, so comparing against it is immune to the truncation risk a raw row-fetch has.
server_count = int(pd.read_json(soda_url(
    FLOOD_EVENTS_RESOURCE,
    **{'$select': 'count(*)', '$where': "flood_start_time > '2025-01-01T00:00:00'"},
))['count'][0])
assert len(flood_data) == server_count, (
    f"Row-limit truncation detected: fetched {len(flood_data)} but server reports {server_count}"
)
print(f"Verified: {len(flood_data)} rows fetched matches server-side count(*) exactly -- not truncated")

Verified: 1303 rows fetched matches server-side count(*) exactly -- not truncated


## Classify every sensor: has events / not yet deployed / silent

Sensors retired before 2025 even started can't have logged a 2025 event regardless of data quality -- excluded the same way `not_yet_deployed` sensors are, via a `retired_before_2025` category, so they don't inflate the "unexplained silence" count. Caveat: `date_removed` is a NYC Open Data field that can lag Directus by weeks in either direction (see [[sensor-name-reconciliation]]), so this is a flag worth spot-checking, not a guaranteed fact -- shown explicitly below rather than silently dropped.

In [5]:
def normalize_name(name):
    return ' '.join(str(name).strip().lower().split())

flood_data['_key'] = flood_data['sensor_name'].apply(normalize_name)
# Same '2026 cutoff' definition tidal_analysis.ipynb uses: an event is "pre-2026" if it
# happened before this year, which for this >2025-01-01 fetch means "happened in 2025".
pre2026_counts = flood_data[flood_data['flood_start_time'] < '2026-01-01'].groupby('_key').size()

all_sensors['_key'] = all_sensors['sensor_name'].apply(normalize_name)
# .map() leaves a sensor with no key in pre2026_counts as NaN (never appeared in the
# events feed at all); .fillna(0) turns that into "had zero events", which is the
# correct reading -- a sensor's absence from flood_data IS a zero-event count, not a
# missing/unknown value.
all_sensors['pre2026_flood_count'] = all_sensors['_key'].map(pre2026_counts).fillna(0)

has_events = all_sensors['pre2026_flood_count'] > 0
not_yet_deployed = all_sensors['date_deployed'] >= '2026-01-01'
# date_removed comes from NYC Open Data, not Directus, and can lag reality by weeks in
# either direction -- FloodNet_Sensor_Locations.ipynb found ~6 sensors where NYC Open
# Data has date_removed set but Directus still lists the device active (see
# sensor-name-reconciliation notes). So this is a flag to review, not a guaranteed
# retirement -- but a sensor removed before 2025 even started couldn't have logged a
# 2025 event regardless of which system is right, so it doesn't belong in "unexplained
# silence" either way.
retired_before_2025 = all_sensors['date_removed'].notna() & (all_sensors['date_removed'] < '2025-01-01')

# 'silent_live' = deployed before 2026, not retired before 2025, but zero logged events
# in 2025 -- the real data-quality question, now asked of every deployed sensor instead
# of just the ~200-sensor tidal/floodplain candidate set.
silent_live_condition = ~has_events & ~not_yet_deployed & ~retired_before_2025
# The four conditions are mutually exclusive and exhaustive over has_events /
# not_yet_deployed / retired_before_2025, so default='unknown' is a safety net that
# should never actually fire -- if it does, one of those three checks has a gap.
all_sensors['status'] = np.select(
    [has_events, not_yet_deployed, retired_before_2025, silent_live_condition],
    ['has_events', 'not_yet_deployed', 'retired_before_2025', 'silent_live'],
    default='unknown',
)
all_sensors['status'].value_counts()

status
silent_live         174
has_events          150
not_yet_deployed     91
Name: count, dtype: int64

## How much bigger is the problem across the full network?

In [6]:
# The two groups this whole notebook is actually about: silent_live_all is the
# unexplained-silence group (what gets tiered and reported below), retired_sensors is
# the small group excluded because retirement already explains their silence.
silent_live_all = all_sensors[all_sensors['status'] == 'silent_live'].copy()
retired_sensors = all_sensors[all_sensors['status'] == 'retired_before_2025'].copy()

print(f"Full network: {len(all_sensors)} sensors")
print(all_sensors['status'].value_counts())
print()
print(f"Silent-live sensors (deployed before 2026, not retired before 2025, zero events in 2025): {len(silent_live_all)}")
print(f"Excluded as retired before 2025 (would've been silent for a trivial reason): {len(retired_sensors)}")
print()
print("Retired-before-2025 sensors, for spot-checking against Directus given the sync-lag caveat above:")
# Sorted by date_removed so the earliest (most confidently "actually gone") removals
# surface first; empty for this run since none of the 6 date_removed sensors predate 2025.
retired_sensors[['sensor_name', 'date_deployed', 'date_removed', 'sensor_status']].sort_values('date_removed')

Full network: 415 sensors
status
silent_live         174
has_events          150
not_yet_deployed     91
Name: count, dtype: int64

Silent-live sensors (deployed before 2026, not retired before 2025, zero events in 2025): 174
Excluded as retired before 2025 (would've been silent for a trivial reason): 0

Retired-before-2025 sensors, for spot-checking against Directus given the sync-lag caveat above:


,sensor_name,date_deployed,date_removed,sensor_status


This is saying that there are 174 sensors that were deployed before 2026, but had no events detected in 2025 (according to NYC Open Data's API endpoint).

## Ruling out a citywide 2025 outage

Before trusting sensor-level gaps, rule out the simplest alternative explanation: that FloodNet's whole pipeline had a bad 2025.

In [7]:
# flood events by year, where n is the number of events and sensors is the number of sensors that detected an event
citywide_by_year = pd.read_json(soda_url(
    FLOOD_EVENTS_RESOURCE,
    **{
        '$select': 'date_extract_y(flood_start_time) as yr, count(*) as n, count(distinct sensor_name) as sensors',
        '$group': 'yr',
        '$order': 'yr',
    },
))
citywide_by_year

,yr,n,sensors
0,2020,2,1
1,2021,58,8
2,2022,142,12
3,2023,362,36
4,2024,662,53
5,2025,678,151
6,2026,625,191


678 flood events were detected in 2025. The likelihood that 174 sensors detected NONE of those 678 floods is low. 

## Per-sensor history and tiering

In [8]:
# Full event history for every sensor (no date filter, no name filter), fetched once and
# filtered locally -- simpler than building a giant IN (...) clause for however many
# hundred sensors turn out to be silent, and avoids any URL-length risk from doing so.
all_history = pd.read_json(soda_url(
    FLOOD_EVENTS_RESOURCE,
    **{'$select': 'sensor_id, sensor_name, flood_start_time', '$limit': 50000},  # actual total is 2,529
))
all_history['yr'] = pd.to_datetime(all_history['flood_start_time']).dt.year

total_count = int(pd.read_json(soda_url(FLOOD_EVENTS_RESOURCE, **{'$select': 'count(*)'}))['count'][0])
assert len(all_history) == total_count, f"Truncated: fetched {len(all_history)}, server has {total_count}"
print(f"Verified: fetched full history ({len(all_history)} events, all sensors/years) matches count(*) exactly")

raw_events = all_history[all_history['sensor_name'].apply(normalize_name).isin(silent_live_all['_key'])].copy()
print(f"{len(raw_events)} historical events found for {raw_events['sensor_name'].nunique()} / {len(silent_live_all)} silent sensors")

Verified: fetched full history (2529 events, all sensors/years) matches count(*) exactly
197 historical events found for 73 / 174 silent sensors


Of the 174 silent sensors, 73 of them detected events either before or after 2025. 

In [9]:
# One row per silent sensor, one column per year -- reindexed so sensors with zero
# events in EVERY year still show up as an all-NaN row instead of dropping out.
year_pivot = (
    raw_events.pivot_table(index='sensor_name', columns='yr', values='sensor_id', aggfunc='count')
    .reindex(silent_live_all['sensor_name'])
)
year_pivot

yr,2020,2021,2022,2023,2024,2026
sensor_name,,,,,,
BX - Louis Niñé Bl/Southern Bl,NaN,NaN,NaN,NaN,NaN,NaN
Q - Craft Ave/Huxley St,NaN,NaN,NaN,NaN,NaN,NaN
BX - Corlear Ave/W 234th St,NaN,NaN,NaN,NaN,NaN,NaN
BX - Lincoln Ave / Bruckner Blvd,NaN,NaN,NaN,NaN,NaN,1.0
Q - Beach 140th St/Cronston Ave,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...
M - W 24th St/12th Ave,NaN,NaN,NaN,NaN,NaN,NaN
BK - Bergen St / 3rd Ave,NaN,NaN,NaN,NaN,NaN,NaN
Q - 77th St/62nd Ave,NaN,NaN,NaN,NaN,NaN,3.0


In [10]:
# Tier by strength of evidence, same logic as tidal_analysis.ipynb:
#   Tier 1: worked before AND after 2025 (single device throughout -- strongest signal)
#   Tier 2: worked before 2025, no 2026 data yet
#   Tier 3: deployed before 2025 (full-year exposure), zero events until 2026
#   Tier 4: deployed during 2025 (partial exposure) -- weaker evidence
pre2025_years = [y for y in year_pivot.columns if y < 2025]
deploy_dates = silent_live_all.set_index('sensor_name')['date_deployed']

# single_device guards against a name being reused for a different physical sensor
# (a redeployment), which could fake "worked before/after" continuity across the gap.
device_counts = raw_events.groupby('sensor_name')['sensor_id'].nunique()

def tier(name):
    row = year_pivot.loc[name] if name in year_pivot.index else None
    has_pre = row[pre2025_years].notna().any() if row is not None else False
    has_post = pd.notna(row[2026]) if row is not None and 2026 in row.index else False
    single_device = device_counts.get(name, 0) <= 1
    deployed = deploy_dates.get(name)
    full_year_2025 = pd.notna(deployed) and deployed < pd.Timestamp('2025-01-01')

    if has_pre and has_post and single_device:
        return 'Tier 1: worked before AND after 2025'
    if has_pre and single_device:
        return 'Tier 2: worked before 2025, no 2026 data yet'
    if full_year_2025:
        return 'Tier 3: live all of 2025 (deployed before 2025), zero events until 2026'
    if pd.notna(deployed) and deployed < pd.Timestamp('2026-01-01'):
        return 'Tier 4: deployed during 2025, only a partial window -- weak evidence'
    return 'Never reported any event, any year'

silent_live_all['gap_tier'] = silent_live_all['sensor_name'].apply(tier)
tier_counts = silent_live_all['gap_tier'].value_counts()
print(tier_counts)
print()
print(f"Tiers 1-3 combined -- sensors with real 2025 exposure and no events (strongest case): "
      f"{tier_counts.filter(like='Tier 1').sum() + tier_counts.filter(like='Tier 2').sum() + tier_counts.filter(like='Tier 3').sum()}")

gap_tier
Tier 3: live all of 2025 (deployed before 2025), zero events until 2026    102
Tier 4: deployed during 2025, only a partial window -- weak evidence        63
Tier 2: worked before 2025, no 2026 data yet                                 5
Tier 1: worked before AND after 2025                                         4
Name: count, dtype: int64

Tiers 1-3 combined -- sensors with real 2025 exposure and no events (strongest case): 111


In [11]:
# Full sensor-level detail for the report, ordered by strength of evidence.
# tier_order maps each label to a sort rank because the tier label TEXT doesn't sort
# into the right order alphabetically ("Never reported..." would sort before all four
# numbered tiers under plain string order).
tier_order = {
    'Tier 1: worked before AND after 2025': 0,
    'Tier 2: worked before 2025, no 2026 data yet': 1,
    'Tier 3: live all of 2025 (deployed before 2025), zero events until 2026': 2,
    'Tier 4: deployed during 2025, only a partial window -- weak evidence': 3,
    'Never reported any event, any year': 4,
}
gap_report = silent_live_all[[
    'sensor_name', 'borough', 'date_deployed', 'sensor_status', 'deploy_type',
    'tidally_influenced', 'in_floodplain', 'gap_tier'
]].copy()
gap_report = gap_report.sort_values('gap_tier', key=lambda s: s.map(tier_order))
gap_report.to_csv('floodnet_full_network_gap_tiers.csv', index=False)
print(f"Wrote floodnet_full_network_gap_tiers.csv ({len(gap_report)} rows)")
gap_report

Wrote floodnet_full_network_gap_tiers.csv (174 rows)


,sensor_name,borough,date_deployed,sensor_status,deploy_type,tidally_influenced,in_floodplain,gap_tier
297,SI - Olympia Blvd/ Mapleton Ave,Staten Island,2023-05-19 00:00:00,good,coastal,Yes,True,Tier 1: worked before AND after 2025
315,BK - Carroll St/4th Av,Brooklyn,2021-07-15 00:00:00,good,pluvial,No,False,Tier 1: worked before AND after 2025
342,BK - Hoyt St/5th St,Brooklyn,2020-10-05 00:00:00,good,pluvial,No,True,Tier 1: worked before AND after 2025
346,BK - Winthrop St/ E 94th St,Brooklyn,2023-05-24 00:00:00,good,pluvial,No,False,Tier 1: worked before AND after 2025
322,Q - Beach 66th St/Thursby Ave,Queens,2023-03-17 00:00:00,good,coastal,Yes,True,"Tier 2: worked before 2025, no 2026 data yet"
...,...,...,...,...,...,...,...,...
329,M - W 42nd St/11th Ave,Manhattan,2025-11-07 12:09:00,good,pluvial,No,True,"Tier 4: deployed during 2025, only a partial w..."
398,BX - Van Cortlandt Park S/Bailey Ave,Bronx,2025-03-27 11:24:00,good,pluvial,No,False,"Tier 4: deployed during 2025, only a partial w..."
406,SI - Castleton Ave / Caroline St,Staten Island,2025-02-19 16:37:00,good,pluvial,No,False,"Tier 4: deployed during 2025, only a partial w..."
409,BK - Bergen St / 3rd Ave,Brooklyn,2025-08-13 12:48:00,good,pluvial,No,False,"Tier 4: deployed during 2025, only a partial w..."


### Conclusions - According to NYC Open Data API
**174 sensors detected NO events in 2025.** Of those:
- 102 sensors were deployed before Jan 1st, 2025, but didn't detect any events until Jan 1st, 2026.
- 5 sensors were deployed AND worked before 2025, but haven't' detected any events yet.
- 4 sensors were deployed and worked before 2025, AND worked in 2026, but didn't detect any events in 2025.
- 63 sensors were deployed at some point during the year in 2025, and didn't detect events in 2025. Checked in the following code block.

In [12]:
# Resolving the open question above: of the 63 Tier 4 sensors (deployed mid-2025,
# only a partial exposure window), how many have since reported in 2026? A sensor with
# a 2026 event proves its device and the detection algorithm both work -- it just
# started reporting after its install date, same pattern as Tier 1/2 but without
# pre-2025 history to compare against.
tier4 = silent_live_all[silent_live_all['gap_tier'].str.startswith('Tier 4')]
tier4_2026 = year_pivot.loc[tier4['sensor_name'], 2026].notna() if 2026 in year_pivot.columns else pd.Series(False, index=tier4['sensor_name'])
print(f"Of {len(tier4)} Tier 4 sensors, {tier4_2026.sum()} have at least one 2026 event; "
      f"{len(tier4) - tier4_2026.sum()} have never recorded any event at all (2025 or 2026 so far)")

Of 63 Tier 4 sensors, 23 have at least one 2026 event; 40 have never recorded any event at all (2025 or 2026 so far)


**Resolved:** 23 of the 63 Tier 4 sensors have a 2026 event -- proof those specific devices and the detection pipeline work, they just started reporting after their install date. The other 40 have never recorded a single event in 2025 or 2026 -- weaker evidence on its own than Tiers 1-3, but not contradicted by anything checked so far.

## Ruling out retirement/hardware failure as the explanation

`retired_before_2025` already excludes sensors NYC Open Data's `date_removed` marks as gone before 2025 started. But `date_removed` is known to lag reality in both directions (see [[sensor-name-reconciliation]] / the sync-lag notes), so it's not sufficient on its own. This section checks the independent signal -- Directus's live `sensor_status` -- against the 174 flagged sensors, to see whether the silence tracks device health rather than being a genuine publishing gap.

In [13]:
# If hardware failure explained the 2025 silence, broken/dead statuses should be
# concentrated in the silent group relative to the rest of the network. They aren't.
broken_statuses = ['dead', 'non-ota', 'needs_sensor', 'hardware_issue', 'needs_ota_update', 'needs_driverail', 'unknown']

is_silent = all_sensors['status'] == 'silent_live'
is_broken = all_sensors['sensor_status'].isin(broken_statuses)

silent_broken_rate = (is_silent & is_broken).sum() / is_silent.sum() * 100
other_broken_rate = (~is_silent & is_broken).sum() / (~is_silent).sum() * 100

print(f"Broken/problem sensor_status rate -- silent group: {silent_broken_rate:.1f}% "
      f"({(is_silent & is_broken).sum()}/{is_silent.sum()})")
print(f"Broken/problem sensor_status rate -- rest of network: {other_broken_rate:.1f}% "
      f"({(~is_silent & is_broken).sum()}/{(~is_silent).sum()})")
print()
print(all_sensors.loc[is_silent, 'sensor_status'].value_counts())

Broken/problem sensor_status rate -- silent group: 10.3% (18/174)
Broken/problem sensor_status rate -- rest of network: 14.9% (36/241)

sensor_status
good                    133
noisy                    15
dead                     11
signal                    5
non-ota                   3
low_charge                2
needs_sensor              2
needs_ota_update          2
reposition_requested      1
Name: count, dtype: int64


In [14]:
# Also check the strongest-evidence sensors specifically (Tier 1/2: proven to have
# worked in some other year) -- if any of these had a date_removed, that would
# undermine the "should have been live through 2025" claim for exactly the sensors
# this report leans on hardest.
tier12 = silent_live_all[silent_live_all['gap_tier'].str.startswith('Tier 1') | silent_live_all['gap_tier'].str.startswith('Tier 2')]
print(f"Tier 1/2 sensors (n={len(tier12)}) -- any date_removed set? {tier12['date_removed'].notna().any()}")
print(tier12[['sensor_name', 'sensor_status', 'date_removed', 'gap_tier']].to_string())

Tier 1/2 sensors (n=9) -- any date_removed set? False
                           sensor_name sensor_status date_removed                                      gap_tier
184       BX - Cross St/Minnieford Ave         noisy          NaT  Tier 2: worked before 2025, no 2026 data yet
197  SI - Lenevar Ave & Drumgoole Rd W          good          NaT  Tier 2: worked before 2025, no 2026 data yet
284               Q - 30th Ave/77th St          good          NaT  Tier 2: worked before 2025, no 2026 data yet
297    SI - Olympia Blvd/ Mapleton Ave          good          NaT          Tier 1: worked before AND after 2025
315             BK - Carroll St/4th Av          good          NaT          Tier 1: worked before AND after 2025
322      Q - Beach 66th St/Thursby Ave          good          NaT  Tier 2: worked before 2025, no 2026 data yet
342                BK - Hoyt St/5th St          good          NaT          Tier 1: worked before AND after 2025
346        BK - Winthrop St/ E 94th St          go

### Conclusion -- retirement and hardware failure ruled out

None of the 174 silent sensors are explained by retirement or broken hardware:
- **0 sensors were retired before 2025 started** (`retired_before_2025` is empty for this run).
- **The only 6 sensors with any `date_removed` at all are the known Directus sync-lag cases** (device still listed active in Directus, not a real retirement) -- and none of them predate 2025.
- **Broken/dead `sensor_status` is actually *less* common in the silent group (10.3%) than in the rest of the network (14.9%)** -- the opposite of what a hardware-failure explanation would predict.
- **The 9 strongest-evidence sensors (Tier 1/2) are 8/9 `'good'` status with zero `date_removed` entries.**

This is not a retirement or hardware problem. The 2025 silence is a genuine gap in what FloodNet published, not a symptom of sensors being decommissioned or broken.

### Final conclusion

This confirms, at the scale of the entire FloodNet network, the same problem `tidal_analysis.ipynb` found in the smaller ~200-sensor tidal/floodplain subset: **111 sensors -- 27% of all 415 deployed sensors -- had real exposure time in 2025 (a full year or part of one) and recorded zero flood events**, despite the network overall logging more events (678) and more distinct reporting sensors (151) in 2025 than any prior complete year.

Checked and ruled out as explanations:
- **API row-limit truncation** -- every raw fetch verified against an independent `count(*)`, exact matches throughout.
- **Sensor retirement** -- checked against both NYC Open Data's `date_removed` and Directus's active-device list.
- **Hardware failure** -- broken-status rate is actually *lower* in the silent group than the rest of the network.

This is a genuine data-quality problem in what FloodNet published to NYC Open Data for 2025, not an artifact of this analysis. Ready to report.